In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import json
import os
from scipy.stats import norm

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

COLORS = {
    'Doc': '#4C72B0', 'Img': '#DD8452', 'Movie': '#55A868',
    'Rec': '#C44E52', 'BGM': '#8172B3'
}
DPI = 300

# Load data
with open('../publication/paper/null_hist_real.json', 'r') as f:
    null_data = json.load(f)

with open('../Data/embedded_DB/Bgm/calibration.json', 'r') as f:
    bgm_cal = json.load(f)

# Load production calibration (5 domains)
with open('../Data/embedded_DB/trichef_calibration.json', 'r') as f:
    trichef_cal = json.load(f)

In [ ]:
# fig03_null_distribution_4domain.png - 2x2 subplot of null distributions

domain_map = [
    ('doc_page', 'Doc',   COLORS['Doc']),
    ('image',    'Img',   COLORS['Img']),
    ('movie',    'Movie', COLORS['Movie']),
    ('music',    'Rec',   COLORS['BGM']),   # music uses BGM color #8172B3
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes_flat = axes.flatten()

for ax, (domain_key, label, color) in zip(axes_flat, domain_map):
    domain_info = null_data[domain_key]
    centers = np.array(domain_info['centers'])
    hist    = np.array(domain_info['hist'])
    mu      = domain_info['mu']
    sigma   = domain_info['sigma']

    # Bar chart of histogram
    bin_width = centers[1] - centers[0] if len(centers) > 1 else 0.01
    ax.bar(centers, hist, width=bin_width * 0.9, color=color, alpha=0.5, label='Null histogram')

    # Fitted normal curve (scaled to match histogram area)
    x_range = np.linspace(centers[0] - 3 * sigma, centers[-1] + 3 * sigma, 400)
    pdf_vals = norm.pdf(x_range, mu, sigma)
    # Scale: histogram is a density (already normalized) or counts?
    # Scale pdf to match the histogram peak
    if hist.max() > 0:
        scale = hist.max() / pdf_vals.max()
    else:
        scale = 1.0
    ax.plot(x_range, pdf_vals * scale, color=color, linewidth=2, label='Normal fit')

    # Vertical lines
    abs_threshold = mu + 1.645 * sigma
    ax.axvline(mu,            color='black', linestyle='--', linewidth=1.5, label=f'mu = {mu:.3f}')
    ax.axvline(abs_threshold, color='red',   linestyle='--', linewidth=1.5, label=f'FAR=0.05 thr = {abs_threshold:.3f}')

    # Text annotations
    ymax = ax.get_ylim()[1] if ax.get_ylim()[1] != 0 else 1
    ax.text(0.03, 0.95, f'mu = {mu:.4f}\nsigma = {sigma:.4f}',
            transform=ax.transAxes, fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

    ax.set_title(f'{label} 도메인 Null 분포', color=color, fontsize=13, fontweight='bold')
    ax.set_xlabel('Cosine Similarity', fontsize=10)
    ax.set_ylabel('Count / Density', fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('도메인별 Null 분포 (Calibration 기반)', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()

out_path = os.path.join(os.path.dirname(os.path.abspath('fig03_calibration_null_dist.ipynb')), 'fig03_null_distribution_4domain.png')
plt.savefig(out_path, dpi=DPI, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

In [ ]:
# fig03_bgm_calibration.png - BGM calibration separation

mu_null      = 0.251384
sigma_null   = 0.08520
music_mean   = 0.5619
separation   = 3.645   # in sigma units

# Estimate music sigma (assume similar spread to null, or derive from separation)
# separation = (music_mean - mu_null) / sigma_null => ~3.645 (given)
# For visualization, use an estimated sigma for the music distribution
sigma_music = sigma_null * 0.9   # slightly narrower, estimated

x = np.linspace(0.0, 0.85, 800)

pdf_null  = norm.pdf(x, mu_null,    sigma_null)
pdf_music = norm.pdf(x, music_mean, sigma_music)

fig, ax = plt.subplots(figsize=(12, 6))

# Fill curves
ax.fill_between(x, pdf_null,  alpha=0.4, color='gray',          label=f'Null 분포  (mu={mu_null:.4f}, sigma={sigma_null:.4f})')
ax.fill_between(x, pdf_music, alpha=0.4, color=COLORS['BGM'],   label=f'Music 분포 (mu={music_mean:.4f}, sigma~{sigma_music:.4f})')

# Outline curves
ax.plot(x, pdf_null,  color='gray',        linewidth=2)
ax.plot(x, pdf_music, color=COLORS['BGM'], linewidth=2)

# Overlap region
overlap = np.minimum(pdf_null, pdf_music)
ax.fill_between(x, overlap, alpha=0.6, color='#FFAA00', label='겹침 영역')

# Vertical lines at means
ax.axvline(mu_null,    color='gray',        linestyle='--', linewidth=1.5)
ax.axvline(music_mean, color=COLORS['BGM'], linestyle='--', linewidth=1.5)

# Annotation: separation arrow
y_arrow = max(pdf_null.max(), pdf_music.max()) * 0.6
ax.annotate(
    '', xy=(music_mean, y_arrow), xytext=(mu_null, y_arrow),
    arrowprops=dict(arrowstyle='<->', color='black', lw=2)
)
ax.text((mu_null + music_mean) / 2, y_arrow * 1.07,
        f'분리 거리\n{separation:.3f} sigma',
        ha='center', va='bottom', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax.set_title('BGM 도메인 Null vs Match 분포 분리', fontsize=14, fontweight='bold')
ax.set_xlabel('Cosine Similarity', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()

out_path2 = os.path.join(os.path.dirname(os.path.abspath('fig03_calibration_null_dist.ipynb')), 'fig03_bgm_calibration.png')
plt.savefig(out_path2, dpi=DPI, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path2}')

In [ ]:
# fig03_confidence_sigmoid.png - Raw score to confidence transformation

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

z = np.linspace(-5, 10, 500)
conf = sigmoid(z / 2.0)

# Domain reference points: (label, color, mu, sigma, ref_raw_score)
# mu and sigma from null_hist_real.json (typical values)
domain_refs = [
    ('Doc',   COLORS['Doc'],   0.17,  0.055, 0.40),   # typical match score
    ('Img',   COLORS['Img'],   0.18,  0.060, 0.45),
    ('Movie', COLORS['Movie'], 0.16,  0.050, 0.38),
    ('Music', COLORS['BGM'],   0.681, 0.085, 0.80),   # music domain is much higher
]

fig, ax = plt.subplots(figsize=(10, 6))

# Main sigmoid curve
ax.plot(z, conf, color='steelblue', linewidth=2.5, label='sigmoid(z/2)')

# Reference lines at confidence = 0.5 and 0.95
ax.axhline(0.5,  color='gray', linestyle=':', linewidth=1, alpha=0.7)
ax.axhline(0.95, color='gray', linestyle=':', linewidth=1, alpha=0.7)
ax.text(9.5, 0.51, '0.5',  fontsize=9, color='gray', va='bottom')
ax.text(9.5, 0.96, '0.95', fontsize=9, color='gray', va='bottom')

# Mark each domain's reference z-score
for label, color, mu, sigma, ref_score in domain_refs:
    z_ref  = (ref_score - mu) / sigma
    c_ref  = sigmoid(z_ref / 2.0)
    # Clamp to plot range
    if -5 <= z_ref <= 10:
        ax.scatter(z_ref, c_ref, color=color, zorder=5, s=80)
        ax.axvline(z_ref, color=color, linestyle='--', linewidth=1, alpha=0.6)
        ax.text(z_ref + 0.15, c_ref - 0.04,
                f'{label}\n(z={z_ref:.1f}, c={c_ref:.2f})',
                fontsize=8, color=color,
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7, edgecolor=color))

ax.set_xlim(-5, 10)
ax.set_ylim(-0.05, 1.05)
ax.set_title('Raw Score -> Confidence 변환 (Sigmoid)', fontsize=14, fontweight='bold')
ax.set_xlabel('Z-score  ( (raw - mu) / sigma )', fontsize=12)
ax.set_ylabel('Confidence (0~1)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()

out_path3 = os.path.join(os.path.dirname(os.path.abspath('fig03_calibration_null_dist.ipynb')), 'fig03_confidence_sigmoid.png')
plt.savefig(out_path3, dpi=DPI, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path3}')

In [ ]:
# fig03_5domain_calibration_params.png — 5도메인 캘리브레이션 파라미터 비교
# trichef_calibration.json (doc_page, image, movie, music) + BGM calibration.json

domains_cal = ['Doc', 'Img', 'Movie', 'Rec', 'BGM']
domain_keys = ['doc_page', 'image', 'movie', 'music', None]  # None = BGM (별도 파일)
domain_colors = [COLORS[d] for d in domains_cal]

# Gather calibration parameters
mu_vals, sigma_vals, threshold_vals, far_vals, n_vals = [], [], [], [], []
for dk in domain_keys:
    if dk is not None:
        cal = trichef_cal[dk]
        mu_vals.append(cal['mu_null'])
        sigma_vals.append(cal['sigma_null'])
        threshold_vals.append(cal['abs_threshold'])
        far_vals.append(cal['far'])
        n_vals.append(cal['N'])
    else:  # BGM
        mu_vals.append(bgm_cal['mu_null'])
        sigma_vals.append(bgm_cal['sigma_null'])
        threshold_vals.append(bgm_cal['p95_null'])
        far_vals.append(0.05)
        n_vals.append(bgm_cal['n_null'] * 100)  # approximate corpus

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('5도메인 캘리브레이션 파라미터 비교', fontsize=16, fontweight='bold', y=0.98)

# (0,0) mu_null comparison
ax = axes[0][0]
bars = ax.bar(domains_cal, mu_vals, color=domain_colors, edgecolor='white', width=0.6)
for b, v in zip(bars, mu_vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.01, f'{v:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('$\\mu_{null}$ (Null 분포 평균)', fontsize=13, fontweight='bold')
ax.set_ylabel('Cosine Similarity', fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# (0,1) sigma_null comparison
ax = axes[0][1]
bars = ax.bar(domains_cal, sigma_vals, color=domain_colors, edgecolor='white', width=0.6)
for b, v in zip(bars, sigma_vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.001, f'{v:.4f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('$\\sigma_{null}$ (Null 분포 표준편차)', fontsize=13, fontweight='bold')
ax.set_ylabel('Standard Deviation', fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# (1,0) abs_threshold comparison
ax = axes[1][0]
bars = ax.bar(domains_cal, threshold_vals, color=domain_colors, edgecolor='white', width=0.6)
for b, v in zip(bars, threshold_vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.01, f'{v:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Absolute Threshold (FAR 기반)', fontsize=13, fontweight='bold')
ax.set_ylabel('Threshold Score', fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# (1,1) FAR & Corpus size
ax = axes[1][1]
x_pos = np.arange(len(domains_cal))
width = 0.35

# FAR bars (left)
bars1 = ax.bar(x_pos - width/2, far_vals, width, color=domain_colors,
               edgecolor='white', alpha=0.7, label='FAR')
for b, v in zip(bars1, far_vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.005, f'{v:.2f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylabel('FAR (False Alarm Rate)', fontsize=11)
ax.set_title('FAR & 코퍼스 크기', fontsize=13, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(domains_cal)
ax.legend(loc='upper left', fontsize=10)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Corpus size as text annotations
ax2 = ax.twinx()
ax2.bar(x_pos + width/2, n_vals, width, color=domain_colors,
        edgecolor='gray', alpha=0.3, label='Corpus N')
for i, v in enumerate(n_vals):
    ax2.text(x_pos[i] + width/2, v + 500, f'{v:,}',
             ha='center', va='bottom', fontsize=8, color='gray')
ax2.set_ylabel('Corpus Size (segments)', fontsize=10, color='gray')
ax2.tick_params(axis='y', labelcolor='gray')
ax2.legend(loc='upper right', fontsize=10)

plt.tight_layout(rect=[0, 0, 1, 0.95])
out_path4 = os.path.join(os.getcwd(), 'fig03_5domain_calibration_params.png')
plt.savefig(out_path4, dpi=DPI, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path4}')

In [ ]:
# fig03_density_overlay_5domain.png — 5도메인 Null Density 오버레이 비교

fig, ax = plt.subplots(figsize=(14, 7))

# 4 domains from null_hist_real.json (실측 히스토그램 + 정규 피팅)
density_domains = [
    ('doc_page', 'Doc',   COLORS['Doc']),
    ('image',    'Img',   COLORS['Img']),
    ('movie',    'Movie', COLORS['Movie']),
    ('music',    'Rec',   COLORS['Rec']),
]

for dk, label, color in density_domains:
    info = null_data['domains'][dk]
    mu, sigma = info['mu'], info['sigma']
    # Plot fitted Gaussian curve
    x = np.linspace(max(0, mu - 4*sigma), mu + 4*sigma, 400)
    pdf = norm.pdf(x, mu, sigma)
    ax.plot(x, pdf, color=color, linewidth=2.5, label=f'{label}  ($\\mu$={mu:.3f}, $\\sigma$={sigma:.3f})')
    ax.fill_between(x, pdf, alpha=0.12, color=color)
    # Mark mu with dot
    ax.scatter([mu], [norm.pdf(mu, mu, sigma)], color=color, s=60, zorder=5, edgecolors='white', linewidths=1)
    # Threshold vertical line
    thr = trichef_cal.get(dk, {}).get('abs_threshold', mu + 1.645*sigma)
    ax.axvline(thr, color=color, linestyle=':', linewidth=1.2, alpha=0.6)

# BGM domain (from separate calibration file, Gaussian approximation)
mu_bgm, sigma_bgm = bgm_cal['mu_null'], bgm_cal['sigma_null']
x_bgm = np.linspace(max(0, mu_bgm - 4*sigma_bgm), mu_bgm + 4*sigma_bgm, 400)
pdf_bgm = norm.pdf(x_bgm, mu_bgm, sigma_bgm)
ax.plot(x_bgm, pdf_bgm, color=COLORS['BGM'], linewidth=2.5, linestyle='--',
        label=f'BGM  ($\\mu$={mu_bgm:.3f}, $\\sigma$={sigma_bgm:.3f})')
ax.fill_between(x_bgm, pdf_bgm, alpha=0.12, color=COLORS['BGM'])
ax.scatter([mu_bgm], [norm.pdf(mu_bgm, mu_bgm, sigma_bgm)], color=COLORS['BGM'],
           s=60, zorder=5, edgecolors='white', linewidths=1)
ax.axvline(bgm_cal['p95_null'], color=COLORS['BGM'], linestyle=':', linewidth=1.2, alpha=0.6)

# Annotations
ax.annotate('Doc/Rec: 고 mu\n(Im축 단독, 텍스트 기반)',
            xy=(0.75, 3.5), fontsize=9, ha='center', color='#555555',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))
ax.annotate('Img/Movie/BGM: 저 mu\n(Re+Im+Z, 크로스모달)',
            xy=(0.2, 8), fontsize=9, ha='center', color='#555555',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

ax.set_title('5도메인 Null Score Density 오버레이 비교', fontsize=15, fontweight='bold')
ax.set_xlabel('Hermitian Cosine Score', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.legend(fontsize=10, loc='upper right')
ax.grid(alpha=0.3)
ax.set_xlim(-0.02, 1.05)

plt.tight_layout()
out_path5 = os.path.join(os.getcwd(), 'fig03_density_overlay_5domain.png')
plt.savefig(out_path5, dpi=DPI, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path5}')